# CPU-only EfficientNet-B0 audit and smoke-test preparation

This notebook performs read-only checks. It does not start training.

In [ ]:
import sys
from pathlib import Path
import torch
from torchvision import datasets

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = ROOT / 'src'
sys.path.insert(0, str(SRC))
from model import CLASS_NAMES, DATASET_ROOT, IMAGE_SIZE, create_model, get_transforms

device = torch.device('cpu')
checkpoint = ROOT / 'models' / 'best_efficientnet_b0.pth'
print('PyTorch:', torch.__version__)
print('CUDA available (ignored):', torch.cuda.is_available())
print('Training device:', device)
print('Dataset path:', DATASET_ROOT)
print('Checkpoint path:', checkpoint)
print('Checkpoint exists:', checkpoint.is_file())
print('Classes:', CLASS_NAMES)
for split in ('train', 'validation', 'test'):
    split_dataset = datasets.ImageFolder(DATASET_ROOT / split)
    print(split, 'images:', len(split_dataset), 'mapping:', split_dataset.class_to_idx)
train_transform, validation_transform = get_transforms(IMAGE_SIZE)
print('Input size:', IMAGE_SIZE)
print('Train preprocessing:', train_transform)
print('Validation preprocessing:', validation_transform)
model = create_model(len(CLASS_NAMES), pretrained=False).to(device)
checkpoint_data = torch.load(checkpoint, map_location='cpu', weights_only=False)
model.load_state_dict(checkpoint_data['model_state_dict'])
with torch.no_grad():
    output = model(torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=device))
print('Checkpoint compatible: PASS')
print('Dummy output shape:', tuple(output.shape))
print('Protected checkpoint will not be overwritten: YES')

In [ ]:
output_dir = ROOT / 'artifacts' / 'finetuning' / 'cpu_smoke_test'
print('Smoke test is prepared but NOT run.')
print('Command:')
print('python src/cpu_experiment_runner.py --epochs 2 --batch-size 16 --num-workers 0 --output-dir artifacts/finetuning/cpu_smoke_test')
print('Output directory:', output_dir)
print('Expected files: best_efficientnet_b0_cpu.pth, training_history.csv, config.json, classification_report.json, confusion_matrix.csv')